In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm.autonotebook import tqdm
import os
from copy import deepcopy
from scipy.optimize import curve_fit
from scipy.stats import t

C:\Users\Administrator\AppData\Local\Temp\ipykernel_38068\1367502170.py:5: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
names = locals()

In [3]:
country_raster_1km_    = rio.open('../results/country-rasters/country-raster-eck4-1km.tif')
country_raster_50km_   = rio.open('../results/country-rasters/country-raster-eck4-50km.tif')
mining_suitability_1km = rio.open('../results/mining-suitability/global-mining-suitability-1km.tif').read(1)
country_raster_50km    = country_raster_50km_.read(1)

In [4]:
def allocate_country(demand_raster, capacity_raster, rarity_raster, radii=[1, 2, 3]):
    i_set, j_set = np.where(demand_raster > 0)
    allocated_extraction = np.zeros_like(demand_raster, dtype=np.float64)
    remained_demand = deepcopy(demand_raster / 1e6)  # Mt
    remained_capacity = deepcopy(capacity_raster.astype(np.float64))

    for r in radii:
        i_set, j_set = np.where(remained_demand > 0)
        demand_values = remained_demand[i_set, j_set]
        sorted_indices = np.argsort(-demand_values)

        for idx in sorted_indices:
            i, j = int(i_set[idx]), int(j_set[idx])
            a_demand = remained_demand[i, j]
            if a_demand <= 0:
                continue

            capacity_slice = remained_capacity[i-r:i+r+1, j-r:j+r+1]
            rarity_slice = rarity_raster[i-r:i+r+1, j-r:j+r+1]
            mask = capacity_slice > 0
            if not np.any(mask):
                continue

            rarity_values = rarity_slice[mask]
            candidate_indices = np.argwhere(mask)
            sorted_order = np.argsort(rarity_values)
            sorted_indices_local = candidate_indices[sorted_order]

            for x, y in sorted_indices_local:
                i2, j2 = i - r + x, j - r + y
                alloc = min(a_demand, capacity_slice[x, y])

                allocated_extraction[i2, j2] += alloc
                remained_capacity[i2, j2] -= alloc
                remained_demand[i, j] -= alloc
                a_demand -= alloc
                if a_demand <= 0:
                    break
                
    return allocated_extraction, remained_demand, remained_capacity

In [ ]:
def strategic_planning(scenario='CON', level_1= 'M', level_2= 'M', level_3= 'M', res='50km'):
    if scenario in ['BAU', 'CE']:
        s_1 = scenario
        s_2 = 'BAU'
    elif scenario == 'CON':
        s_1 = 'BAU'
        s_2 = 'CON'
    elif scenario == 'IAP':
        s_1 = 'CE'
        s_2 = 'CON'

    country_raster_ = rio.open(f'../results/country-rasters/country-raster-eck4-{res}.tif')
    economy_raster_ = rio.open(f'../results/economy-rasters/economy-raster-eck4-{res}.tif')
    country_raster  = country_raster_.read(1)
    economy_raster  = economy_raster_.read(1)

    sand_demand_raster      = rio.open(f'../results/uncertainty-analysis/sand-demand-2023-2050-{s_1}-{res}-{level_1}.tif').read(1)
    gravel_demand_raster    = rio.open(f'../results/uncertainty-analysis/gravel-demand-2023-2050-{s_1}-{res}-{level_1}.tif').read(1)
    limestone_demand_raster = rio.open(f'../results/uncertainty-analysis/limestone-demand-2023-2050-{s_1}-{res}-{level_1}.tif').read(1)
    total_demand_raster     = sand_demand_raster + gravel_demand_raster + limestone_demand_raster

    extraction_capacity  = rio.open(f'../results/uncertainty-analysis/extraction-capacity-{s_2}-{res}-{level_2}{level_3}.tif').read(1)
    RWR = rio.open(f'../results/area-of-habitat-{res}/rarity-weighted-richness-vertebrates-{res}.tif').read(1)

    final_extraction_raster     = np.zeros(country_raster_.shape, dtype=np.float64)
    remained_demand_global   = np.zeros(country_raster_.shape, dtype=np.float64)
    remained_capacity_global = np.zeros(country_raster_.shape, dtype=np.float64)

    country_codes = np.unique(country_raster)[1:]

    for code in tqdm(country_codes, desc="Allocating by country"):
        mask = country_raster == code
        
        country_demand = total_demand_raster * mask
        country_capacity = extraction_capacity * mask
        country_rarity = RWR * mask
        
        alloc, remain_demand, remain_capacity = allocate_country(
            country_demand, country_capacity, country_rarity
        )

        final_extraction_raster     += alloc
        remained_demand_global   += remain_demand
        remained_capacity_global += remain_capacity

    for code in country_codes:
        mask = (country_raster == code)
        country_remained_demand   = remained_demand_global[mask].sum()
        country_remained_capacity = remained_capacity_global[mask].sum()

        if country_remained_demand == 0:
            continue

        if country_remained_capacity > 0:
            # print(code)
            cap_country = remained_capacity_global[mask]
            w_country = cap_country / cap_country.sum()
            add_local = w_country * min(country_remained_demand, country_remained_capacity)

            final_extraction_raster[mask]  += add_local
            remained_capacity_global[mask] -= add_local

            country_remained_demand -= add_local.sum()

        if country_remained_demand == 0:
            remained_demand_global[mask] = 0


    remained_capacity_global = np.where(economy_raster > 0, remained_capacity_global, 0)
    w_global = remained_capacity_global / remained_capacity_global.sum()
    aaa = (np.sum(total_demand_raster) - np.sum(final_extraction_raster) * 1e6) / 1e6
    # final_extraction_raster += w_global * np.sum(remained_demand_global)
    final_extraction_raster += w_global * aaa    

    coverage_2 = 100 * np.sum(final_extraction_raster) / np.sum(total_demand_raster/1e6)
    print(f"✅ Global coverage ({scenario}-{level_1}{level_2}{level_3}): {np.sum(final_extraction_raster)/1e3:.3f}/{np.sum(total_demand_raster/1e9):.3f} Gt ({coverage_2:.1f}%)")

    output_path = f'../results/uncertainty-analysis/allocated-extraction-2023-2050-{scenario}-{res}-{level_1}{level_2}{level_3}.tif'
    with rio.open(
        output_path,
        'w',
        driver='GTiff',
        height=country_raster.shape[0],
        width=country_raster.shape[1],
        count=1,
        dtype=final_extraction_raster.dtype,
        crs=country_raster_.crs,
        transform=country_raster_.transform,
        compress='lzw',
    ) as dst:
        dst.write(final_extraction_raster, 1)

    return final_extraction_raster

In [ ]:
for scenario in ['CON', 'IAP']:
    for l_1 in ['L', 'M', 'H']:
        for l_2 in ['L', 'M', 'H']:
            for l_3 in ['L', 'M', 'H']:
                gg = strategic_planning(scenario, l_1, l_2, l_3)